In [5]:
from z3 import *
from utils import * 
import sympy as sp
import numpy as np
from itertools import product
from fractions import Fraction
import random

In [6]:
def create_z3_variables(E):
    return {
        (u, v): Real(f"x_{u}_{v}")
        for u, v in E
    }

def create_random_B(E, n):
    L = sp.zeros(n, n)

    for u, v in E:
        # random rational number
        numerator = random.randint(-1000, 1000)
        denominator = random.randint(1, 1000)
        L[u-1, v-1] = sp.Rational(numerator, denominator)

    I = sp.eye(n)
    return (I - L).inv().T

def create_equation(i, j, n, x_vars, B):

    expr = B[i-1,j-1]   # constant term

    for k in range(n):
        if (k+1, i) in x_vars and B[k,j-1]!=0:
            expr -= x_vars[(k+1, i)] * B[k,j-1]

    return expr

def create_variables(edge_list):
    return {(u, v): sp.Symbol(f"m_{u}_{v}") for u, v in edge_list}


def get_M_matrix(edge_list, n):
    vars = create_variables(edge_list)
    M = sp.zeros(n)

    for (u, v), var in vars.items():
        M[u-1, v-1] = var

    return M, list(vars.values())

def get_L_matrix(edge_list, n, low=-1.0, high=1.0):
    L = np.zeros((n, n))

    for u, v in edge_list:
        L[u-1, v-1] = np.random.uniform(low, high)

    return L

def get_J(B2, n):
    B2 = set(B2)
    return [(i, j) for i in range(1, n+1)
                   for j in range(i+1, n+1)
                   if (i, j) not in B2]

def get_K(B1, n):
    symmetric_B1 = set(B1) | {(j, i) for i, j in B1}
    return [(i, j)
            for i in range(1, n+1)
            for j in range(1, n+1)
            if i == j or (i, j) in symmetric_B1]

def map_pairs(JK):
    return [((a, c), (b, d)) for ((a, b), (c, d)) in JK]

def filter_pairs(JK, A):
    return [((a, b), (c, d)) 
            for ((a, b), (c, d)) in JK
            if A[a-1, b-1] != 0 and A[c-1, d-1] != 0]

def create_equations(JK, A):
    return [
        A[a-1, b-1] * A[c-1, d-1]
        for ((a, b), (c, d)) in JK
    ]

def get_both_linear_equations(JK, A):
    return [
        [A[a-1, b-1] , A[c-1, d-1]]
        for ((a, b), (c, d)) in JK
    ]

def get_solver(E1,E2,B1,B2,n):

    L = get_L_matrix(E1, n)
    M, vars = get_M_matrix(E2, n)
    I = np.eye(n)
    A = (I-M).T @ np.linalg.inv(I - L).T
    J = get_J(B2,n)
    K = get_K(B1,n)
    JK = map_pairs(list(product(J,K)))
    JK_reduced = filter_pairs(JK, A)
    JK_reduced

    constant_indices = [
        (i+1, j+1)
        for i in range(A.rows)
        for j in range(A.cols)
        if not (A[i, j].free_symbols & set(vars))
    ]

    x = create_z3_variables(E2)
    B = create_random_B(E1, n)

    s = Solver()

    for (a,b),(c,d) in JK_reduced:

        if (a,b) in constant_indices:
            if (c,d) in constant_indices:
                return False
            else:
                s.add(create_equation(c, d, n, x, B)==0)
        elif (c,d) in constant_indices:
            s.add(create_equation(a,b,n,x,B)==0)
        else:    
            f = create_equation(a, b, n, x, B)
            g = create_equation(c, d, n, x, B)
            b = Bool(f"choose_{(a,b,c,d)}")
            s.add(If(b, f == 0, g == 0))

    return s.check()

In [15]:
n = 3

# Generate DAGs
G1 = nx.DiGraph()
G2 = nx.DiGraph()
V = range(1,n+1)
G1.add_nodes_from(V)
G2.add_nodes_from(V)

# Edge Sets
E1 = [(1,2),(2,3)]
E2 = [(1,2),(1,3)]
B1 = [(1,3)]
B2 = [(1,3)]

# Add Edges
G1.add_edges_from(E1)
G2.add_edges_from(E2)

In [122]:
num_vertex = 25
num_confounding = 25

G1 = random_dag(num_vertex,random.random())
B1 = random_confounding(num_vertex,num_confounding)
G2 = random_dag(num_vertex,random.random())
B2 = random_confounding(num_vertex,num_confounding)
E1 = G1.edges()
E2 = G2.edges()

In [123]:
get_solver(E1,E2,B1,B2,num_vertex)

False

In [124]:
import time
import csv
import numpy as np
from utils import *

In [126]:
def runtime_record(n, filename):

    times = []

    for i in range(500):
        num_confounding = random.randint(0, n * (n - 1) // 2)

        G1 = random_dag(n,random.random())
        B1 = random_confounding(n,num_confounding)
        G2 = random_dag(n,random.random())
        B2 = random_confounding(n,num_confounding)

        start = time.perf_counter()

        get_solver(G1.edges(),G2.edges(),B1,B2,num_vertex)

        runtime = time.perf_counter() - start
        times.append(runtime)

    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["run", "runtime_seconds"])
        
        for i, t in enumerate(times):
            writer.writerow([i + 1, t])

    mean_time = np.mean(times)
    std_time = np.std(times)

    print(f"Mean runtime: {mean_time:.6f}s")
    print(f"Std runtime: {std_time:.6f}s")

In [127]:
runtime_record(6, "timerecord_z3/runtime_6.csv")

Mean runtime: 0.029238s
Std runtime: 0.011534s


In [128]:
runtime_record(8, "timerecord_z3/runtime_8.csv")

Mean runtime: 0.049957s
Std runtime: 0.033479s


In [129]:
runtime_record(10, "timerecord_z3/runtime_10.csv")

Mean runtime: 0.090233s
Std runtime: 0.080509s


In [130]:
runtime_record(12, "timerecord_z3/runtime_12.csv")

Mean runtime: 0.149868s
Std runtime: 0.159448s


In [131]:
runtime_record(14, "timerecord_z3/runtime_14.csv")

Mean runtime: 0.319520s
Std runtime: 0.405208s


In [132]:
runtime_record(16, "timerecord_z3/runtime_16.csv")

Mean runtime: 0.543239s
Std runtime: 0.693510s


In [133]:
runtime_record(18, "timerecord_z3/runtime_18.csv")

Mean runtime: 0.966108s
Std runtime: 1.333381s


In [134]:
runtime_record(20, "timerecord_z3/runtime_20.csv")

Mean runtime: 1.713686s
Std runtime: 2.523234s


In [135]:
runtime_record(22, "timerecord_z3/runtime_22.csv")

Mean runtime: 2.759754s
Std runtime: 4.124411s


In [136]:
runtime_record(24, "timerecord_z3/runtime_24.csv")

Mean runtime: 4.767474s
Std runtime: 7.224174s


In [152]:
runtime_record(26, "timerecord_z3/runtime_26.csv")

IndexError: index 25 is out of bounds for axis 1 with size 25